# Per-video diagnostic - which videos are easiest / hardest

Evaluates every video separately (its own frames only) with Ultralytics' own metric,
for train, val and test, and prints the 5 best and 5 worst videos per split. This shows
what the model finds easy vs hard and explains the val/test difference.

Runs on GPU. ~1-1.5 h total (train is the slow part) - you can stop after val+test.
Upload `eval_exclude.txt` to Drive root first.

## 1. Setup

In [ ]:
!nvidia-smi
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/iadd_subset_v4
!unzip -q "/content/drive/MyDrive/iadd_subset_v4.zip" -d /content

## 2. Remove the leaked frames (same clean set as the reported eval)

In [ ]:
import os, glob
DS = '/content/iadd_subset_v4'
for entry in [l.strip() for l in open('/content/drive/MyDrive/eval_exclude.txt') if l.strip()]:
    sp, stem = entry.split('/', 1)
    for sub, ext in (('images', '.jpg'), ('labels', '.txt')):
        p = f'{DS}/{sub}/{sp}/{stem}{ext}'
        if os.path.exists(p): os.remove(p)
for c in glob.glob(f'{DS}/labels/*.cache'):
    os.remove(c)
print('leaked frames removed')

## 3. Per-video evaluation

For each video, symlink only its frames into a temp folder and validate it. Records
images, instances, condition (D/N/R/A), mAP50 and mAP50-95. Saves a CSV to Drive.

In [ ]:
from ultralytics import YOLO
import os, glob, yaml, shutil
from collections import defaultdict
import pandas as pd

DS   = '/content/iadd_subset_v4'
RUNS = '/content/drive/MyDrive/iadd_runs'
best = YOLO(f'{RUNS}/run5/weights/best.pt')
NAMES = ['person','car','motorcycle','bus','truck','traffic_light']

def per_video(split):
    groups = defaultdict(list)
    for p in glob.glob(f'{DS}/images/{split}/*.jpg'):
        groups[os.path.basename(p).split('__')[0]].append(p)
    rows = []
    for k, (vid, imgs) in enumerate(sorted(groups.items()), 1):
        TMP = '/content/pv_tmp'
        shutil.rmtree(TMP, ignore_errors=True)
        os.makedirs(f'{TMP}/images/v'); os.makedirs(f'{TMP}/labels/v')
        inst = 0
        for img in imgs:
            b = os.path.basename(img)
            os.symlink(img, f'{TMP}/images/v/{b}')
            lb = f'{DS}/labels/{split}/' + b[:-4] + '.txt'
            if os.path.exists(lb):
                os.symlink(lb, f'{TMP}/labels/v/' + b[:-4] + '.txt')
                inst += sum(1 for _ in open(lb))
        yaml.safe_dump({'path': TMP, 'train': 'images/v', 'val': 'images/v',
                        'nc': 6, 'names': NAMES}, open(f'{TMP}/data.yaml', 'w'))
        m = best.val(data=f'{TMP}/data.yaml', split='val', verbose=False, plots=False,
                     project='/content/pvruns', name='tmp', exist_ok=True)
        rows.append({'split': split, 'video': vid, 'cond': vid.split('_')[-1],
                     'images': len(imgs), 'instances': inst,
                     'mAP50': round(float(m.box.map50), 4), 'mAP50_95': round(float(m.box.map), 4)})
        if k % 10 == 0:
            print(f'  {split}: {k}/{len(groups)}', flush=True)
    return rows

all_rows = []
for split in ('val', 'test', 'train'):     # val+test first (the important ones)
    print(f'=== evaluating {split} videos ===', flush=True)
    all_rows += per_video(split)

df = pd.DataFrame(all_rows)
df.to_csv(f'{RUNS}/pervideo_run5.csv', index=False)
print('\nsaved -> iadd_runs/pervideo_run5.csv')

## 4. Best and worst 5 videos per split

In [ ]:
for split in ('train', 'val', 'test'):
    d = df[df.split == split].sort_values('mAP50')
    print(f'\n===== {split.upper()}  ({len(d)} videos, mean mAP50={d.mAP50.mean():.3f}) =====')
    print('  WORST 10 (hardest):')
    for _, r in d.head(10).iterrows():
        print(f"    {r.video:<16} cond={r.cond}  img={r.images:>4}  inst={r.instances:>5}  mAP50={r.mAP50:.3f}")
    print('  BEST 10 (easiest):')
    for _, r in d.tail(10).iloc[::-1].iterrows():
        print(f"    {r.video:<16} cond={r.cond}  img={r.images:>4}  inst={r.instances:>5}  mAP50={r.mAP50:.3f}")

print('\n===== mean mAP50 by condition (D=day N=night R=rain A=other) =====')
print(df.groupby(['split', 'cond'])['mAP50'].mean().round(3).unstack())